# 🤖 EarnKaro Price Comparison Telegram Bot

**Run Order:** Cell 1 → 2 → 3 → 5 (skip 4 for now) → 7 → 8

In [ ]:
# === Cell 1: Install Dependencies ===
!pip install requests==2.31.0 beautifulsoup4==4.12.2 lxml==4.9.3 rapidfuzz==3.5.2     python-telegram-bot==20.6 gspread==5.12.0 google-auth==2.23.4 fake-useragent==1.4.0 -q
print("✅ Dependencies installed")

In [ ]:
# === Cell 2: Mount Google Drive & Set Working Directory ===
import os, sys
from google.colab import drive

drive.mount("/content/drive")

BOT_DIR = "/content/drive/MyDrive/earnkaro_bot"
os.makedirs(BOT_DIR, exist_ok=True)
os.chdir(BOT_DIR)
if BOT_DIR not in sys.path:
    sys.path.insert(0, BOT_DIR)

print(f"✅ Working directory: {os.getcwd()}")

In [ ]:
# === Cell 3: Write All Source Files to Drive ===
import base64, os

files_b64 = {
    "config.py": "IyBjb25maWcucHkg4oCUIEFsbCBjb25zdGFudHMsIGhlYWRlcnMsIENTUyBzZWxlY3RvcnMKCiMgUGxhdGZvcm0gZG9tYWluIGRldGVjdGlvbgpQTEFURk9STV9ET01BSU5TID0gewogICAgImFtYXpvbi5pbiI6ICJhbWF6b24iLAogICAgImZsaXBrYXJ0LmNvbSI6ICJmbGlwa2FydCIsCiAgICAibXludHJhLmNvbSI6ICJteW50cmEiLAogICAgImhhbWFyYW1hbGwuY29tIjogImhhbWFyYW1hbGwiLAp9CgojIFJlcXVlc3QgaGVhZGVycyBwZXIgcGxhdGZvcm0KSEVBREVSUyA9IHsKICAgICJhbWF6b24iOiB7CiAgICAgICAgIlVzZXItQWdlbnQiOiAiTW96aWxsYS81LjAgKFdpbmRvd3MgTlQgMTAuMDsgV2luNjQ7IHg2NCkgQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgQ2hyb21lLzEyMC4wLjAuMCBTYWZhcmkvNTM3LjM2IiwKICAgICAgICAiQWNjZXB0LUxhbmd1YWdlIjogImVuLUlOLGVuO3E9MC45IiwKICAgIH0sCiAgICAiZmxpcGthcnQiOiB7CiAgICAgICAgIlVzZXItQWdlbnQiOiAiTW96aWxsYS81LjAgKFdpbmRvd3MgTlQgMTAuMDsgV2luNjQ7IHg2NCkgQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgQ2hyb21lLzEyMC4wLjAuMCBTYWZhcmkvNTM3LjM2IiwKICAgIH0sCiAgICAibXludHJhIjogewogICAgICAgICJVc2VyLUFnZW50IjogIk1vemlsbGEvNS4wIChXaW5kb3dzIE5UIDEwLjA7IFdpbjY0OyB4NjQpIEFwcGxlV2ViS2l0LzUzNy4zNiAoS0hUTUwsIGxpa2UgR2Vja28pIENocm9tZS8xMjAuMC4wLjAgU2FmYXJpLzUzNy4zNiIsCiAgICB9LAogICAgImhhbWFyYW1hbGwiOiB7CiAgICAgICAgIlVzZXItQWdlbnQiOiAiTW96aWxsYS81LjAgKFdpbmRvd3MgTlQgMTAuMDsgV2luNjQ7IHg2NCkgQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgQ2hyb21lLzEyMC4wLjAuMCBTYWZhcmkvNTM3LjM2IiwKICAgIH0sCn0KCiMgTWF0Y2hpbmcgc2NvcmUgdGhyZXNob2xkcwpTQ09SRV9FWEFDVCA9IDc1ICAgICAgIyBTaG93IGFzIOKchSBFeGFjdCBtYXRjaApTQ09SRV9TSU1JTEFSID0gNDAgICAgIyBTaG93IGFzIOKaoO+4jyBTaW1pbGFyIHByb2R1Y3QKIyBCZWxvdyA0MCA9IOKdjCBOb3QgZm91bmQKCiMgU2VhcmNoIHJlc3VsdCBjb3VudCB0byBldmFsdWF0ZSBwZXIgcGxhdGZvcm0KVE9QX05fUkVTVUxUUyA9IDUKCiMgRGVsYXlzIGJldHdlZW4gcmVxdWVzdHMgKHNlY29uZHMpIHRvIGF2b2lkIHJhdGUgbGltaXRpbmcKUkVRVUVTVF9ERUxBWSA9IDEuNQoKIyBHb29nbGUgU2hlZXQgY29uZmlnClNIRUVUX05BTUUgPSAiRWFybkthcm8gQ29tbWlzc2lvbnMiClNIRUVUX1RBQiA9ICJDYXRlZ29yeSBDb21taXNzaW9uIgpDT01NSVNTSU9OX0NPTF9DQVRFR09SWSA9IDAgICMgQ29sdW1uIEEKQ09NTUlTU0lPTl9DT0xfUEVSQ0VOVCA9IDEgICAjIENvbHVtbiBCCgojIENTUyBTZWxlY3RvcnMg4oCUIEFtYXpvbgpBTUFaT05fU0VMRUNUT1JTID0gewogICAgImJyYW5kIjogIiNieWxpbmVJbmZvIHNwYW4iLAogICAgInRpdGxlIjogIiNwcm9kdWN0VGl0bGUiLAogICAgInByaWNlX3ByaW1hcnkiOiAiI3ByaWNlYmxvY2tfb3VycHJpY2UiLAogICAgInByaWNlX3dob2xlIjogIi5hLXByaWNlLXdob2xlIiwKICAgICJwcmljZV9kZWFsIjogIiNwcmljZWJsb2NrX2RlYWxwcmljZSIsCiAgICAiYnJlYWRjcnVtYiI6ICIjd2F5ZmluZGluZy1icmVhZGNydW1ic19mZWF0dXJlX2RpdiIsCiAgICAic2VhcmNoX3Jlc3VsdCI6ICdkaXZbZGF0YS1jb21wb25lbnQtdHlwZT0icy1zZWFyY2gtcmVzdWx0Il0nLAogICAgInNlYXJjaF90aXRsZSI6ICIuYS1zaXplLW1lZGl1bSwgLmEtc2l6ZS1iYXNlLXBsdXMiLAogICAgInNlYXJjaF9wcmljZSI6ICIuYS1wcmljZS13aG9sZSIsCiAgICAic2VhcmNoX2xpbmsiOiAiYS5hLWxpbmstbm9ybWFsIiwKfQoKIyBDU1MgU2VsZWN0b3JzIOKAlCBGbGlwa2FydApGTElQS0FSVF9TRUxFQ1RPUlMgPSB7CiAgICAidGl0bGUiOiAiLl8zNUt5RDYsIC5CX051Q0kiLAogICAgInByaWNlIjogIi5fMzBqZXEzLl8xNkprNmQiLAogICAgImJyZWFkY3J1bWIiOiAiLl8yd2hLYW8iLAp9CgojIENTUyBTZWxlY3RvcnMg4oCUIE15bnRyYQpNWU5UUkFfU0VMRUNUT1JTID0gewogICAgImJyYW5kIjogIi5wZHAtdGl0bGUiLAogICAgInRpdGxlIjogIi5wZHAtbmFtZSIsCiAgICAicHJpY2UiOiAiLnBkcC1wcmljZSBzdHJvbmcsIC5wZHAtbXJwIiwKfQoKIyBDU1MgU2VsZWN0b3JzIOKAlCBIYW1hcmEgTWFsbCAodG8gYmUgdXBkYXRlZCBhZnRlciBsaXZlIGluc3BlY3Rpb24pCkhBTUFSQU1BTExfU0VMRUNUT1JTID0gewogICAgImJyYW5kIjogIiIsCiAgICAidGl0bGUiOiAiIiwKICAgICJwcmljZSI6ICIiLAogICAgImNhdGVnb3J5IjogIiIsCn0KCiMgUGxhdGZvcm0gZW1vamkgbWFwClBMQVRGT1JNX0VNT0pJID0gewogICAgImFtYXpvbiI6ICLwn5+gIiwKICAgICJmbGlwa2FydCI6ICLwn5+hIiwKICAgICJteW50cmEiOiAi8J+UtSIsCiAgICAiaGFtYXJhbWFsbCI6ICLwn4+qIiwKfQoKIyBTdG9wIHdvcmRzIGZvciBrZXl3b3JkIGV4dHJhY3Rpb24KU1RPUF9XT1JEUyA9IHsiZm9yIiwgIndpdGgiLCAiYW5kIiwgIm9mIiwgInRoZSIsICJpbiIsICJhIiwgImFuIiwgIi0ifQo=",
    "utils.py": "IyB1dGlscy5weSDigJQgU2hhcmVkIGhlbHBlcnMKCmltcG9ydCByZQpmcm9tIHVybGxpYi5wYXJzZSBpbXBvcnQgdXJscGFyc2UKZnJvbSBjb25maWcgaW1wb3J0IFBMQVRGT1JNX0RPTUFJTlMsIFNUT1BfV09SRFMKCgpkZWYgZGV0ZWN0X3BsYXRmb3JtKHVybDogc3RyKSAtPiBzdHI6CiAgICAiIiJQYXJzZSBVUkwgZG9tYWluIGFuZCByZXR1cm4gcGxhdGZvcm0gbmFtZS4gUmFpc2VzIFZhbHVlRXJyb3IgaWYgdW5rbm93bi4iIiIKICAgIGRvbWFpbiA9IHVybHBhcnNlKHVybCkubmV0bG9jLmxvd2VyKCkucmVwbGFjZSgid3d3LiIsICIiKQogICAgZm9yIGtleSwgcGxhdGZvcm0gaW4gUExBVEZPUk1fRE9NQUlOUy5pdGVtcygpOgogICAgICAgIGlmIGtleSBpbiBkb21haW46CiAgICAgICAgICAgIHJldHVybiBwbGF0Zm9ybQogICAgcmFpc2UgVmFsdWVFcnJvcihmIlVucmVjb2duaXNlZCBkb21haW46IHtkb21haW59IikKCgpkZWYgbm9ybWFsaXplX3F1YW50aXR5KHRleHQ6IHN0cikgLT4gc3RyOgogICAgIiIiU3RhbmRhcmRpemUgcXVhbnRpdHkgc3RyaW5ncyBmb3IgY29tcGFyaXNvbi4iIiIKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiAiIgogICAgdGV4dCA9IHRleHQubG93ZXIoKS5zdHJpcCgpLnJlcGxhY2UoIiAiLCAiIikKCiAgICAjIENvbnZlcnQga2cgdG8gZwogICAga2dfbWF0Y2ggPSByZS5tYXRjaChyIl4oXGQrKD86XC5cZCspPylrZyQiLCB0ZXh0KQogICAgaWYga2dfbWF0Y2g6CiAgICAgICAgZ3JhbXMgPSBpbnQoZmxvYXQoa2dfbWF0Y2guZ3JvdXAoMSkpICogMTAwMCkKICAgICAgICByZXR1cm4gZiJ7Z3JhbXN9ZyIKCiAgICAjIE5vcm1hbGl6ZSAiZ20iIHRvICJnIgogICAgdGV4dCA9IHJlLnN1YihyIihcZClnbSQiLCByIlwxZyIsIHRleHQpCgogICAgIyBOb3JtYWxpemUgInBhY2sgb2YgTiIgLyAieE4iIHRvICJOcGFjayIKICAgIHBhY2tfbWF0Y2ggPSByZS5tYXRjaChyInBhY2tvZihcZCspIiwgdGV4dCkKICAgIGlmIHBhY2tfbWF0Y2g6CiAgICAgICAgcmV0dXJuIGYie3BhY2tfbWF0Y2guZ3JvdXAoMSl9cGFjayIKICAgIHhfbWF0Y2ggPSByZS5tYXRjaChyIngoXGQrKSQiLCB0ZXh0KQogICAgaWYgeF9tYXRjaDoKICAgICAgICByZXR1cm4gZiJ7eF9tYXRjaC5ncm91cCgxKX1wYWNrIgoKICAgIHJldHVybiB0ZXh0CgoKZGVmIGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZSh0aXRsZTogc3RyKSAtPiBzdHI6CiAgICAiIiJVc2UgcmVnZXggdG8gZmluZCBxdWFudGl0eSBwYXR0ZXJucyBpbiBhIHByb2R1Y3QgdGl0bGUuIiIiCiAgICBpZiBub3QgdGl0bGU6CiAgICAgICAgcmV0dXJuICIiCiAgICBwYXR0ZXJucyA9IFsKICAgICAgICByIihcZCsoPzpcLlxkKyk/XHMqKD86bWx8bHxrZ3xnfGdtfG96KSlcYiIsICAjIDI1MG1sLCAxLjVrZywgMjAwZywgMTAwZ20KICAgICAgICByIihwYWNrXHMqb2ZccypcZCspIiwgICAgICAgICAgICAgICAgICAgICAgICAgICAjIHBhY2sgb2YgMgogICAgICAgIHIiXGIoeFxkKylcYiIsICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgeDMKICAgIF0KICAgIGZvciBwYXR0ZXJuIGluIHBhdHRlcm5zOgogICAgICAgIG1hdGNoID0gcmUuc2VhcmNoKHBhdHRlcm4sIHRpdGxlLCByZS5JR05PUkVDQVNFKQogICAgICAgIGlmIG1hdGNoOgogICAgICAgICAgICByZXR1cm4gbWF0Y2guZ3JvdXAoMSkuc3RyaXAoKQogICAgcmV0dXJuICIiCgoKZGVmIGNsZWFuX3ByaWNlKHByaWNlX3N0cjogc3RyKSAtPiBpbnQ6CiAgICAiIiJTdHJpcCBjdXJyZW5jeSBzeW1ib2xzLCBjb21tYXMsIGRlY2ltYWxzLiBSZXR1cm4gaW50ZWdlciBwcmljZSBvciAwLiIiIgogICAgaWYgbm90IHByaWNlX3N0cjoKICAgICAgICByZXR1cm4gMAogICAgIyBGaW5kIHRoZSBmaXJzdCBudW1iZXIgKHdpdGggb3B0aW9uYWwgZGVjaW1hbHMpCiAgICBtYXRjaCA9IHJlLnNlYXJjaChyIihcZFtcZCxdKig/OlwuXGQrKT8pIiwgcHJpY2Vfc3RyKQogICAgaWYgbm90IG1hdGNoOgogICAgICAgIHJldHVybiAwCiAgICBjbGVhbmVkID0gbWF0Y2guZ3JvdXAoMSkucmVwbGFjZSgiLCIsICIiKQogICAgdHJ5OgogICAgICAgIHJldHVybiBpbnQoZmxvYXQoY2xlYW5lZCkpCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICByZXR1cm4gMAoKCmRlZiBleHRyYWN0X2tleXdvcmRzKHRpdGxlOiBzdHIsIGJyYW5kOiBzdHIpIC0+IGxpc3Q6CiAgICAiIiJFeHRyYWN0IG1lYW5pbmdmdWwga2V5d29yZHMgZnJvbSB0aXRsZSwgcmVtb3ZpbmcgYnJhbmQsIHN0b3Agd29yZHMsIHF1YW50aXR5LiIiIgogICAgaWYgbm90IHRpdGxlOgogICAgICAgIHJldHVybiBbXQogICAgIyBSZW1vdmUgYnJhbmQgZnJvbSB0aXRsZQogICAgY2xlYW5lZCA9IHJlLnN1YihyZS5lc2NhcGUoYnJhbmQpLCAiIiwgdGl0bGUsIGZsYWdzPXJlLklHTk9SRUNBU0UpLnN0cmlwKCkKICAgICMgUmVtb3ZlIHF1YW50aXR5IHN0cmluZ3MKICAgIHF0eSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShjbGVhbmVkKQogICAgaWYgcXR5OgogICAgICAgIGNsZWFuZWQgPSBjbGVhbmVkLnJlcGxhY2UocXR5LCAiIikKICAgICMgVG9rZW5pemUsIGxvd2VyY2FzZSwgZmlsdGVyCiAgICB3b3JkcyA9IGNsZWFuZWQubG93ZXIoKS5zcGxpdCgpCiAgICBrZXl3b3JkcyA9IFt3LnN0cmlwKCIsLSgpIikgZm9yIHcgaW4gd29yZHMgaWYgdy5zdHJpcCgiLC0oKSIpIGFuZCB3LnN0cmlwKCIsLSgpIikgbm90IGluIFNUT1BfV09SRFNdCiAgICByZXR1cm4ga2V5d29yZHMK",
    "extractor.py": "IyBleHRyYWN0b3IucHkg4oCUIFNvdXJjZSBwYWdlIHNjcmFwZXIKCmltcG9ydCB0aW1lCmltcG9ydCByZXF1ZXN0cwpmcm9tIGJzNCBpbXBvcnQgQmVhdXRpZnVsU291cAoKZnJvbSBjb25maWcgaW1wb3J0IEhFQURFUlMsIEFNQVpPTl9TRUxFQ1RPUlMsIEZMSVBLQVJUX1NFTEVDVE9SUywgTVlOVFJBX1NFTEVDVE9SUywgSEFNQVJBTUFMTF9TRUxFQ1RPUlMKZnJvbSB1dGlscyBpbXBvcnQgZGV0ZWN0X3BsYXRmb3JtLCBleHRyYWN0X3F1YW50aXR5X2Zyb21fdGl0bGUsIGV4dHJhY3Rfa2V5d29yZHMsIGNsZWFuX3ByaWNlCgoKZGVmIGV4dHJhY3RfcHJvZHVjdCh1cmw6IHN0cikgLT4gZGljdDoKICAgICIiIk1haW4gZW50cnk6IGZldGNoIFVSTCwgZGV0ZWN0IHBsYXRmb3JtLCBleHRyYWN0IHN0cnVjdHVyZWQgcHJvZHVjdCBkYXRhLiIiIgogICAgcHJvZHVjdCA9IHsKICAgICAgICAicGxhdGZvcm0iOiAiIiwKICAgICAgICAidXJsIjogdXJsLAogICAgICAgICJicmFuZCI6ICIiLAogICAgICAgICJuYW1lIjogIiIsCiAgICAgICAgInF1YW50aXR5IjogIiIsCiAgICAgICAgImtleXdvcmRzIjogW10sCiAgICAgICAgImNhdGVnb3J5IjogIiIsCiAgICAgICAgInByaWNlIjogMCwKICAgICAgICAic2VhcmNoX3F1ZXJ5IjogIiIsCiAgICAgICAgIndhcm5pbmdzIjogW10sCiAgICB9CgogICAgdHJ5OgogICAgICAgIHBsYXRmb3JtID0gZGV0ZWN0X3BsYXRmb3JtKHVybCkKICAgICAgICBwcm9kdWN0WyJwbGF0Zm9ybSJdID0gcGxhdGZvcm0KCiAgICAgICAgaGVhZGVycyA9IEhFQURFUlMuZ2V0KHBsYXRmb3JtLCB7fSkKICAgICAgICByZXNwID0gcmVxdWVzdHMuZ2V0KHVybCwgaGVhZGVycz1oZWFkZXJzLCB0aW1lb3V0PTE1KQogICAgICAgIHJlc3AucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgc291cCA9IEJlYXV0aWZ1bFNvdXAocmVzcC50ZXh0LCAibHhtbCIpCgogICAgICAgIGV4dHJhY3RvcnMgPSB7CiAgICAgICAgICAgICJhbWF6b24iOiBleHRyYWN0X2Zyb21fYW1hem9uLAogICAgICAgICAgICAiZmxpcGthcnQiOiBleHRyYWN0X2Zyb21fZmxpcGthcnQsCiAgICAgICAgICAgICJteW50cmEiOiBleHRyYWN0X2Zyb21fbXludHJhLAogICAgICAgICAgICAiaGFtYXJhbWFsbCI6IGV4dHJhY3RfZnJvbV9oYW1hcmFtYWxsLAogICAgICAgIH0KCiAgICAgICAgZXh0cmFjdG9yID0gZXh0cmFjdG9ycy5nZXQocGxhdGZvcm0pCiAgICAgICAgaWYgZXh0cmFjdG9yOgogICAgICAgICAgICBkYXRhID0gZXh0cmFjdG9yKHNvdXAsIHVybCkKICAgICAgICAgICAgcHJvZHVjdC51cGRhdGUoZGF0YSkKCiAgICAgICAgIyBGaWxsIGRlcml2ZWQgZmllbGRzCiAgICAgICAgaWYgcHJvZHVjdFsibmFtZSJdIGFuZCBub3QgcHJvZHVjdFsicXVhbnRpdHkiXToKICAgICAgICAgICAgcHJvZHVjdFsicXVhbnRpdHkiXSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShwcm9kdWN0WyJuYW1lIl0pCiAgICAgICAgaWYgcHJvZHVjdFsibmFtZSJdIGFuZCBwcm9kdWN0WyJicmFuZCJdOgogICAgICAgICAgICBwcm9kdWN0WyJrZXl3b3JkcyJdID0gZXh0cmFjdF9rZXl3b3Jkcyhwcm9kdWN0WyJuYW1lIl0sIHByb2R1Y3RbImJyYW5kIl0pCiAgICAgICAgcHJvZHVjdFsic2VhcmNoX3F1ZXJ5Il0gPSBidWlsZF9zZWFyY2hfcXVlcnkocHJvZHVjdCkKCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJvZHVjdFsid2FybmluZ3MiXS5hcHBlbmQoZiJFeHRyYWN0aW9uIGVycm9yOiB7c3RyKGUpfSIpCgogICAgcmV0dXJuIHByb2R1Y3QKCgpkZWYgZXh0cmFjdF9mcm9tX2FtYXpvbihzb3VwLCB1cmwpIC0+IGRpY3Q6CiAgICAiIiJTY3JhcGUgYW4gQW1hem9uLmluIHByb2R1Y3QgcGFnZS4iIiIKICAgIGRhdGEgPSB7Indhcm5pbmdzIjogW119CgogICAgIyBCcmFuZAogICAgdHJ5OgogICAgICAgIGJyYW5kX2VsID0gc291cC5zZWxlY3Rfb25lKEFNQVpPTl9TRUxFQ1RPUlNbImJyYW5kIl0pCiAgICAgICAgaWYgYnJhbmRfZWw6CiAgICAgICAgICAgIGRhdGFbImJyYW5kIl0gPSBicmFuZF9lbC5nZXRfdGV4dChzdHJpcD1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgRmFsbGJhY2s6IGZpcnN0IHdvcmQgb2YgdGl0bGUKICAgICAgICAgICAgdGl0bGVfZWwgPSBzb3VwLnNlbGVjdF9vbmUoQU1BWk9OX1NFTEVDVE9SU1sidGl0bGUiXSkKICAgICAgICAgICAgaWYgdGl0bGVfZWw6CiAgICAgICAgICAgICAgICBkYXRhWyJicmFuZCJdID0gdGl0bGVfZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkuc3BsaXQoKVswXQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgiYnJhbmQgc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIFRpdGxlCiAgICB0cnk6CiAgICAgICAgdGl0bGVfZWwgPSBzb3VwLnNlbGVjdF9vbmUoQU1BWk9OX1NFTEVDVE9SU1sidGl0bGUiXSkKICAgICAgICBkYXRhWyJuYW1lIl0gPSB0aXRsZV9lbC5nZXRfdGV4dChzdHJpcD1UcnVlKSBpZiB0aXRsZV9lbCBlbHNlICIiCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGRhdGFbIndhcm5pbmdzIl0uYXBwZW5kKCJ0aXRsZSBzZWxlY3RvciBmYWlsZWQiKQoKICAgICMgUHJpY2Ug4oCUIHRyeSBtdWx0aXBsZSBzZWxlY3RvcnMgaW4gb3JkZXIKICAgIHRyeToKICAgICAgICBwcmljZV90ZXh0ID0gIiIKICAgICAgICBmb3Igc2VsX2tleSBpbiBbInByaWNlX3ByaW1hcnkiLCAicHJpY2Vfd2hvbGUiLCAicHJpY2VfZGVhbCJdOgogICAgICAgICAgICBlbCA9IHNvdXAuc2VsZWN0X29uZShBTUFaT05fU0VMRUNUT1JTW3NlbF9rZXldKQogICAgICAgICAgICBpZiBlbDoKICAgICAgICAgICAgICAgIHByaWNlX3RleHQgPSBlbC5nZXRfdGV4dChzdHJpcD1UcnVlKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBkYXRhWyJwcmljZSJdID0gY2xlYW5fcHJpY2UocHJpY2VfdGV4dCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoInByaWNlIHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBDYXRlZ29yeSDigJQgYnJlYWRjcnVtYiBzZWNvbmQtdG8tbGFzdCBpdGVtCiAgICB0cnk6CiAgICAgICAgYnJlYWRjcnVtYiA9IHNvdXAuc2VsZWN0X29uZShBTUFaT05fU0VMRUNUT1JTWyJicmVhZGNydW1iIl0pCiAgICAgICAgaWYgYnJlYWRjcnVtYjoKICAgICAgICAgICAgaXRlbXMgPSBicmVhZGNydW1iLnNlbGVjdCgiYSIpCiAgICAgICAgICAgIGlmIGxlbihpdGVtcykgPj0gMjoKICAgICAgICAgICAgICAgIGRhdGFbImNhdGVnb3J5Il0gPSBpdGVtc1stMl0uZ2V0X3RleHQoc3RyaXA9VHJ1ZSkKICAgICAgICAgICAgZWxpZiBpdGVtczoKICAgICAgICAgICAgICAgIGRhdGFbImNhdGVnb3J5Il0gPSBpdGVtc1stMV0uZ2V0X3RleHQoc3RyaXA9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoImNhdGVnb3J5IHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBRdWFudGl0eQogICAgZGF0YVsicXVhbnRpdHkiXSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShkYXRhLmdldCgibmFtZSIsICIiKSkKCiAgICByZXR1cm4gZGF0YQoKCmRlZiBleHRyYWN0X2Zyb21fZmxpcGthcnQoc291cCwgdXJsKSAtPiBkaWN0OgogICAgIiIiU2NyYXBlIGEgRmxpcGthcnQgcHJvZHVjdCBwYWdlLiIiIgogICAgZGF0YSA9IHsid2FybmluZ3MiOiBbXX0KCiAgICAjIEJyYW5kIOKAlCBicmVhZGNydW1iIHNlY29uZCBpdGVtLCBvciBmaXJzdCB3b3JkIG9mIHRpdGxlCiAgICB0cnk6CiAgICAgICAgYnJlYWRjcnVtYnMgPSBzb3VwLnNlbGVjdChGTElQS0FSVF9TRUxFQ1RPUlNbImJyZWFkY3J1bWIiXSkKICAgICAgICBpZiBsZW4oYnJlYWRjcnVtYnMpID49IDI6CiAgICAgICAgICAgIGRhdGFbImJyYW5kIl0gPSBicmVhZGNydW1ic1sxXS5nZXRfdGV4dChzdHJpcD1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHRpdGxlX2VsID0gc291cC5zZWxlY3Rfb25lKEZMSVBLQVJUX1NFTEVDVE9SU1sidGl0bGUiXSkKICAgICAgICAgICAgaWYgdGl0bGVfZWw6CiAgICAgICAgICAgICAgICBkYXRhWyJicmFuZCJdID0gdGl0bGVfZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkuc3BsaXQoKVswXQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgiYnJhbmQgc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIFRpdGxlCiAgICB0cnk6CiAgICAgICAgdGl0bGVfZWwgPSBzb3VwLnNlbGVjdF9vbmUoRkxJUEtBUlRfU0VMRUNUT1JTWyJ0aXRsZSJdKQogICAgICAgIGRhdGFbIm5hbWUiXSA9IHRpdGxlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpIGlmIHRpdGxlX2VsIGVsc2UgIiIKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoInRpdGxlIHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBQcmljZQogICAgdHJ5OgogICAgICAgIHByaWNlX2VsID0gc291cC5zZWxlY3Rfb25lKEZMSVBLQVJUX1NFTEVDVE9SU1sicHJpY2UiXSkKICAgICAgICBkYXRhWyJwcmljZSJdID0gY2xlYW5fcHJpY2UocHJpY2VfZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkpIGlmIHByaWNlX2VsIGVsc2UgMAogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgicHJpY2Ugc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIENhdGVnb3J5IOKAlCBicmVhZGNydW1iIHNlY29uZC10by1sYXN0CiAgICB0cnk6CiAgICAgICAgYnJlYWRjcnVtYnMgPSBzb3VwLnNlbGVjdChGTElQS0FSVF9TRUxFQ1RPUlNbImJyZWFkY3J1bWIiXSkKICAgICAgICBpZiBsZW4oYnJlYWRjcnVtYnMpID49IDI6CiAgICAgICAgICAgIGRhdGFbImNhdGVnb3J5Il0gPSBicmVhZGNydW1ic1stMl0uZ2V0X3RleHQoc3RyaXA9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoImNhdGVnb3J5IHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBRdWFudGl0eQogICAgZGF0YVsicXVhbnRpdHkiXSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShkYXRhLmdldCgibmFtZSIsICIiKSkKCiAgICByZXR1cm4gZGF0YQoKCmRlZiBleHRyYWN0X2Zyb21fbXludHJhKHNvdXAsIHVybCkgLT4gZGljdDoKICAgICIiIlNjcmFwZSBhIE15bnRyYSBwcm9kdWN0IHBhZ2UuIiIiCiAgICBkYXRhID0geyJ3YXJuaW5ncyI6IFtdfQoKICAgICMgQnJhbmQKICAgIHRyeToKICAgICAgICBicmFuZF9lbCA9IHNvdXAuc2VsZWN0X29uZShNWU5UUkFfU0VMRUNUT1JTWyJicmFuZCJdKQogICAgICAgIGRhdGFbImJyYW5kIl0gPSBicmFuZF9lbC5nZXRfdGV4dChzdHJpcD1UcnVlKSBpZiBicmFuZF9lbCBlbHNlICIiCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGRhdGFbIndhcm5pbmdzIl0uYXBwZW5kKCJicmFuZCBzZWxlY3RvciBmYWlsZWQiKQoKICAgICMgVGl0bGUKICAgIHRyeToKICAgICAgICB0aXRsZV9lbCA9IHNvdXAuc2VsZWN0X29uZShNWU5UUkFfU0VMRUNUT1JTWyJ0aXRsZSJdKQogICAgICAgIGRhdGFbIm5hbWUiXSA9IHRpdGxlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpIGlmIHRpdGxlX2VsIGVsc2UgIiIKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoInRpdGxlIHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBQcmljZQogICAgdHJ5OgogICAgICAgIHByaWNlX2VsID0gc291cC5zZWxlY3Rfb25lKE1ZTlRSQV9TRUxFQ1RPUlNbInByaWNlIl0pCiAgICAgICAgZGF0YVsicHJpY2UiXSA9IGNsZWFuX3ByaWNlKHByaWNlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpKSBpZiBwcmljZV9lbCBlbHNlIDAKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGF0YVsid2FybmluZ3MiXS5hcHBlbmQoInByaWNlIHNlbGVjdG9yIGZhaWxlZCIpCgogICAgIyBDYXRlZ29yeSDigJQgcGFyc2UgZnJvbSBVUkwgc2x1ZyAoTXludHJhIFVSTHM6IC9icmFuZC9wcm9kdWN0L2NhdGVnb3J5LykKICAgIHRyeToKICAgICAgICBwYXJ0cyA9IFtwIGZvciBwIGluIHVybC5zcGxpdCgiLyIpIGlmIHBdCiAgICAgICAgaWYgbGVuKHBhcnRzKSA+PSA0OgogICAgICAgICAgICBkYXRhWyJjYXRlZ29yeSJdID0gcGFydHNbM10ucmVwbGFjZSgiLSIsICIgIikudGl0bGUoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgiY2F0ZWdvcnkgc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIFF1YW50aXR5CiAgICBkYXRhWyJxdWFudGl0eSJdID0gZXh0cmFjdF9xdWFudGl0eV9mcm9tX3RpdGxlKGRhdGEuZ2V0KCJuYW1lIiwgIiIpKQoKICAgIHJldHVybiBkYXRhCgoKZGVmIGV4dHJhY3RfZnJvbV9oYW1hcmFtYWxsKHNvdXAsIHVybCkgLT4gZGljdDoKICAgICIiIlNjcmFwZSBhIEhhbWFyYSBNYWxsIHByb2R1Y3QgcGFnZS4gU2VsZWN0b3JzIFRCRCDigJQgbmVlZHMgbGl2ZSBpbnNwZWN0aW9uLiIiIgogICAgZGF0YSA9IHsid2FybmluZ3MiOiBbXX0KCiAgICAjIEF0dGVtcHQgZ2VuZXJpYyBleHRyYWN0aW9uIHVzaW5nIGNvbW1vbiBwYXR0ZXJucwogICAgdHJ5OgogICAgICAgICMgVHJ5IGNvbW1vbiB0aXRsZSBzZWxlY3RvcnMKICAgICAgICBmb3Igc2VsIGluIFsiaDEucHJvZHVjdC10aXRsZSIsICJoMS5wcm9kdWN0LW5hbWUiLCAiaDEiLCAiLnByb2R1Y3QtdGl0bGUiLCAiLnByb2R1Y3QtbmFtZSJdOgogICAgICAgICAgICBlbCA9IHNvdXAuc2VsZWN0X29uZShzZWwpCiAgICAgICAgICAgIGlmIGVsIGFuZCBlbC5nZXRfdGV4dChzdHJpcD1UcnVlKToKICAgICAgICAgICAgICAgIGRhdGFbIm5hbWUiXSA9IGVsLmdldF90ZXh0KHN0cmlwPVRydWUpCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGlmIG5vdCBkYXRhLmdldCgibmFtZSIpOgogICAgICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgidGl0bGUgc2VsZWN0b3IgZmFpbGVkIOKAlCBuZWVkcyBsaXZlIGluc3BlY3Rpb24iKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgidGl0bGUgc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIFRyeSBjb21tb24gcHJpY2Ugc2VsZWN0b3JzCiAgICB0cnk6CiAgICAgICAgZm9yIHNlbCBpbiBbIi5wcm9kdWN0LXByaWNlIiwgIi5wcmljZSIsICIuY3VycmVudC1wcmljZSIsICJzcGFuLnByaWNlIl06CiAgICAgICAgICAgIGVsID0gc291cC5zZWxlY3Rfb25lKHNlbCkKICAgICAgICAgICAgaWYgZWw6CiAgICAgICAgICAgICAgICBkYXRhWyJwcmljZSJdID0gY2xlYW5fcHJpY2UoZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkpCiAgICAgICAgICAgICAgICBicmVhawogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBkYXRhWyJ3YXJuaW5ncyJdLmFwcGVuZCgicHJpY2Ugc2VsZWN0b3IgZmFpbGVkIikKCiAgICAjIEJyYW5kIOKAlCBmaXJzdCB3b3JkIG9mIHRpdGxlIGFzIGZhbGxiYWNrCiAgICBpZiBkYXRhLmdldCgibmFtZSIpIGFuZCBub3QgZGF0YS5nZXQoImJyYW5kIik6CiAgICAgICAgZGF0YVsiYnJhbmQiXSA9IGRhdGFbIm5hbWUiXS5zcGxpdCgpWzBdCgogICAgIyBRdWFudGl0eQogICAgZGF0YVsicXVhbnRpdHkiXSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShkYXRhLmdldCgibmFtZSIsICIiKSkKCiAgICByZXR1cm4gZGF0YQoKCmRlZiBidWlsZF9zZWFyY2hfcXVlcnkocHJvZHVjdDogZGljdCkgLT4gc3RyOgogICAgIiIiQ29tYmluZSBicmFuZCArIGNvcmUgbmFtZSArIHF1YW50aXR5IGludG8gYSBzZWFyY2ggc3RyaW5nIChtYXggNjAgY2hhcnMpLiIiIgogICAgcGFydHMgPSBbXQogICAgaWYgcHJvZHVjdC5nZXQoImJyYW5kIik6CiAgICAgICAgcGFydHMuYXBwZW5kKHByb2R1Y3RbImJyYW5kIl0pCiAgICBpZiBwcm9kdWN0LmdldCgia2V5d29yZHMiKToKICAgICAgICBwYXJ0cy5leHRlbmQocHJvZHVjdFsia2V5d29yZHMiXVs6NV0pICAjIExpbWl0IHRvIHRvcCBrZXl3b3JkcwogICAgaWYgcHJvZHVjdC5nZXQoInF1YW50aXR5Iik6CiAgICAgICAgcGFydHMuYXBwZW5kKHByb2R1Y3RbInF1YW50aXR5Il0pCiAgICBxdWVyeSA9ICIgIi5qb2luKHBhcnRzKQogICAgcmV0dXJuIHF1ZXJ5Wzo2MF0uc3RyaXAoKQo=",
    "searcher.py": "IyBzZWFyY2hlci5weSDigJQgQ3Jvc3MtcGxhdGZvcm0gc2VhcmNoCgppbXBvcnQgdGltZQppbXBvcnQgcmUKaW1wb3J0IGpzb24KaW1wb3J0IHJlcXVlc3RzCmZyb20gdXJsbGliLnBhcnNlIGltcG9ydCBxdW90ZV9wbHVzCmZyb20gYnM0IGltcG9ydCBCZWF1dGlmdWxTb3VwCgpmcm9tIGNvbmZpZyBpbXBvcnQgSEVBREVSUywgQU1BWk9OX1NFTEVDVE9SUywgVE9QX05fUkVTVUxUUywgUkVRVUVTVF9ERUxBWQoKCmRlZiBzZWFyY2hfYW1hem9uKHF1ZXJ5OiBzdHIpIC0+IGxpc3Q6CiAgICAiIiJTZWFyY2ggQW1hem9uLmluIGFuZCByZXR1cm4gdG9wIHJlc3VsdHMuIiIiCiAgICB0cnk6CiAgICAgICAgdXJsID0gZiJodHRwczovL3d3dy5hbWF6b24uaW4vcz9rPXtxdW90ZV9wbHVzKHF1ZXJ5KX0iCiAgICAgICAgcmVzcCA9IHJlcXVlc3RzLmdldCh1cmwsIGhlYWRlcnM9SEVBREVSU1siYW1hem9uIl0sIHRpbWVvdXQ9MTUpCiAgICAgICAgcmVzcC5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICBzb3VwID0gQmVhdXRpZnVsU291cChyZXNwLnRleHQsICJseG1sIikKCiAgICAgICAgcmVzdWx0cyA9IFtdCiAgICAgICAgY2FyZHMgPSBzb3VwLnNlbGVjdChBTUFaT05fU0VMRUNUT1JTWyJzZWFyY2hfcmVzdWx0Il0pCiAgICAgICAgZm9yIGNhcmQgaW4gY2FyZHNbOlRPUF9OX1JFU1VMVFNdOgogICAgICAgICAgICB0aXRsZV9lbCA9IGNhcmQuc2VsZWN0X29uZShBTUFaT05fU0VMRUNUT1JTWyJzZWFyY2hfdGl0bGUiXSkKICAgICAgICAgICAgcHJpY2VfZWwgPSBjYXJkLnNlbGVjdF9vbmUoQU1BWk9OX1NFTEVDVE9SU1sic2VhcmNoX3ByaWNlIl0pCiAgICAgICAgICAgIGxpbmtfZWwgPSBjYXJkLnNlbGVjdF9vbmUoQU1BWk9OX1NFTEVDVE9SU1sic2VhcmNoX2xpbmsiXSkKCiAgICAgICAgICAgIGlmIG5vdCB0aXRsZV9lbDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICB0aXRsZSA9IHRpdGxlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpCiAgICAgICAgICAgIHByaWNlX3RleHQgPSBwcmljZV9lbC5nZXRfdGV4dChzdHJpcD1UcnVlKSBpZiBwcmljZV9lbCBlbHNlICIwIgogICAgICAgICAgICAjIENsZWFuIHByaWNlIOKAlCByZW1vdmUgY29tbWFzLCB0YWtlIGludGVnZXIKICAgICAgICAgICAgcHJpY2UgPSAwCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHByaWNlID0gaW50KHJlLnN1YihyIlteXGRdIiwgIiIsIHByaWNlX3RleHQpKQogICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgICAgIHByb2RfdXJsID0gIiIKICAgICAgICAgICAgaWYgbGlua19lbCBhbmQgbGlua19lbC5nZXQoImhyZWYiKToKICAgICAgICAgICAgICAgIGhyZWYgPSBsaW5rX2VsWyJocmVmIl0KICAgICAgICAgICAgICAgIHByb2RfdXJsID0gaHJlZiBpZiBocmVmLnN0YXJ0c3dpdGgoImh0dHAiKSBlbHNlIGYiaHR0cHM6Ly93d3cuYW1hem9uLmlue2hyZWZ9IgoKICAgICAgICAgICAgcmVzdWx0cy5hcHBlbmQoeyJ0aXRsZSI6IHRpdGxlLCAicHJpY2UiOiBwcmljZSwgInVybCI6IHByb2RfdXJsfSkKCiAgICAgICAgcmV0dXJuIHJlc3VsdHMKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIFtdCgoKZGVmIHNlYXJjaF9mbGlwa2FydChxdWVyeTogc3RyKSAtPiBsaXN0OgogICAgIiIiU2VhcmNoIEZsaXBrYXJ0IGFuZCByZXR1cm4gdG9wIHJlc3VsdHMuIiIiCiAgICB0cnk6CiAgICAgICAgdXJsID0gZiJodHRwczovL3d3dy5mbGlwa2FydC5jb20vc2VhcmNoP3E9e3F1b3RlX3BsdXMocXVlcnkpfSIKICAgICAgICByZXNwID0gcmVxdWVzdHMuZ2V0KHVybCwgaGVhZGVycz1IRUFERVJTWyJmbGlwa2FydCJdLCB0aW1lb3V0PTE1KQogICAgICAgIHJlc3AucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgc291cCA9IEJlYXV0aWZ1bFNvdXAocmVzcC50ZXh0LCAibHhtbCIpCgogICAgICAgIHJlc3VsdHMgPSBbXQogICAgICAgICMgRmxpcGthcnQgc2VhcmNoIHJlc3VsdCBjYXJkcyDigJQgdHJ5IG11bHRpcGxlIGtub3duIGNvbnRhaW5lciBzZWxlY3RvcnMKICAgICAgICBjYXJkcyA9IHNvdXAuc2VsZWN0KCJkaXYuXzFBdFZiRSwgZGl2Ll8xeEhHdEssIGRpdi5zbEFWVjQiKQogICAgICAgIGZvciBjYXJkIGluIGNhcmRzWzpUT1BfTl9SRVNVTFRTICogMl06ICAjIE92ZXItc2VsZWN0LCBmaWx0ZXIgbGF0ZXIKICAgICAgICAgICAgdGl0bGVfZWwgPSBjYXJkLnNlbGVjdF9vbmUoImEuSVJwd1RhLCBhLnMxUTlycywgZGl2Ll80clIwMVQsIGEuXzJycHdxSSIpCiAgICAgICAgICAgIHByaWNlX2VsID0gY2FyZC5zZWxlY3Rfb25lKCJkaXYuXzMwamVxMywgZGl2Ll8yNWIxOGMiKQogICAgICAgICAgICBsaW5rX2VsID0gY2FyZC5zZWxlY3Rfb25lKCJhLklScHdUYSwgYS5zMVE5cnMsIGEuXzJycHdxSSwgYS5fMWZRWkVLIikKCiAgICAgICAgICAgIGlmIG5vdCB0aXRsZV9lbDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgICAgICB0aXRsZSA9IHRpdGxlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpCiAgICAgICAgICAgIHByaWNlID0gMAogICAgICAgICAgICBpZiBwcmljZV9lbDoKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwcmljZSA9IGludChyZS5zdWIociJbXlxkXSIsICIiLCBwcmljZV9lbC5nZXRfdGV4dChzdHJpcD1UcnVlKSkpCiAgICAgICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgICAgICBwYXNzCgogICAgICAgICAgICBwcm9kX3VybCA9ICIiCiAgICAgICAgICAgIGlmIGxpbmtfZWwgYW5kIGxpbmtfZWwuZ2V0KCJocmVmIik6CiAgICAgICAgICAgICAgICBocmVmID0gbGlua19lbFsiaHJlZiJdCiAgICAgICAgICAgICAgICBwcm9kX3VybCA9IGhyZWYgaWYgaHJlZi5zdGFydHN3aXRoKCJodHRwIikgZWxzZSBmImh0dHBzOi8vd3d3LmZsaXBrYXJ0LmNvbXtocmVmfSIKCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsidGl0bGUiOiB0aXRsZSwgInByaWNlIjogcHJpY2UsICJ1cmwiOiBwcm9kX3VybH0pCgogICAgICAgICAgICBpZiBsZW4ocmVzdWx0cykgPj0gVE9QX05fUkVTVUxUUzoKICAgICAgICAgICAgICAgIGJyZWFrCgogICAgICAgIHJldHVybiByZXN1bHRzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBbXQoKCmRlZiBzZWFyY2hfbXludHJhKHF1ZXJ5OiBzdHIpIC0+IGxpc3Q6CiAgICAiIiJTZWFyY2ggTXludHJhLiBUcmllcyBKU09OIGluIHNjcmlwdCB0YWcgZmlyc3QsIGZhbGxzIGJhY2sgdG8gSFRNTC4iIiIKICAgIHRyeToKICAgICAgICBzZWFyY2hfc2x1ZyA9IHF1b3RlX3BsdXMocXVlcnkpLnJlcGxhY2UoIisiLCAiLSIpCiAgICAgICAgdXJsID0gZiJodHRwczovL3d3dy5teW50cmEuY29tL3tzZWFyY2hfc2x1Z30iCiAgICAgICAgcmVzcCA9IHJlcXVlc3RzLmdldCh1cmwsIGhlYWRlcnM9SEVBREVSU1sibXludHJhIl0sIHRpbWVvdXQ9MTUpCiAgICAgICAgcmVzcC5yYWlzZV9mb3Jfc3RhdHVzKCkKCiAgICAgICAgcmVzdWx0cyA9IFtdCgogICAgICAgICMgVHJ5IGV4dHJhY3RpbmcgSlNPTiBmcm9tIHNjcmlwdCB0YWcgKHdpbmRvdy5fX215eCBvciBfX0lOSVRJQUxfU1RBVEVfXykKICAgICAgICBqc29uX21hdGNoID0gcmUuc2VhcmNoKAogICAgICAgICAgICByJyg/OndpbmRvd1wuX19teXhccyo9fHdpbmRvd1wuX19JTklUSUFMX1NUQVRFX19ccyo9KVxzKih7Lis/fSk7P1xzKjwvc2NyaXB0PicsCiAgICAgICAgICAgIHJlc3AudGV4dCwgcmUuRE9UQUxMCiAgICAgICAgKQogICAgICAgIGlmIGpzb25fbWF0Y2g6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKGpzb25fbWF0Y2guZ3JvdXAoMSkpCiAgICAgICAgICAgICAgICAjIE5hdmlnYXRlIHRvIHByb2R1Y3QgbGlzdCDigJQgc3RydWN0dXJlIHZhcmllcwogICAgICAgICAgICAgICAgcHJvZHVjdHMgPSBbXQogICAgICAgICAgICAgICAgaWYgInNlYXJjaERhdGEiIGluIGRhdGE6CiAgICAgICAgICAgICAgICAgICAgcHJvZHVjdHMgPSBkYXRhWyJzZWFyY2hEYXRhIl0uZ2V0KCJyZXN1bHRzIiwge30pLmdldCgicHJvZHVjdHMiLCBbXSkKICAgICAgICAgICAgICAgIGVsaWYgInJlc3VsdHMiIGluIGRhdGE6CiAgICAgICAgICAgICAgICAgICAgcHJvZHVjdHMgPSBkYXRhWyJyZXN1bHRzIl0uZ2V0KCJwcm9kdWN0cyIsIFtdKQoKICAgICAgICAgICAgICAgIGZvciBwIGluIHByb2R1Y3RzWzpUT1BfTl9SRVNVTFRTXToKICAgICAgICAgICAgICAgICAgICB0aXRsZSA9IGYie3AuZ2V0KCdicmFuZCcsICcnKX0ge3AuZ2V0KCdwcm9kdWN0TmFtZScsICcnKX0iLnN0cmlwKCkKICAgICAgICAgICAgICAgICAgICBwcmljZSA9IHAuZ2V0KCJwcmljZSIsIDApIG9yIHAuZ2V0KCJkaXNjb3VudGVkUHJpY2UiLCAwKQogICAgICAgICAgICAgICAgICAgIHByb2RfdXJsID0gZiJodHRwczovL3d3dy5teW50cmEuY29tL3twLmdldCgnbGFuZGluZ1BhZ2VVcmwnLCAnJyl9IgogICAgICAgICAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsidGl0bGUiOiB0aXRsZSwgInByaWNlIjogaW50KHByaWNlKSwgInVybCI6IHByb2RfdXJsfSkKCiAgICAgICAgICAgICAgICBpZiByZXN1bHRzOgogICAgICAgICAgICAgICAgICAgIHJldHVybiByZXN1bHRzCiAgICAgICAgICAgIGV4Y2VwdCAoanNvbi5KU09ORGVjb2RlRXJyb3IsIEtleUVycm9yKToKICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgIyBGYWxsYmFjazogcGFyc2UgSFRNTAogICAgICAgIHNvdXAgPSBCZWF1dGlmdWxTb3VwKHJlc3AudGV4dCwgImx4bWwiKQogICAgICAgIGNhcmRzID0gc291cC5zZWxlY3QoIi5wcm9kdWN0LWJhc2UsIC5yZXN1bHRzLWJhc2UgbGkiKQogICAgICAgIGZvciBjYXJkIGluIGNhcmRzWzpUT1BfTl9SRVNVTFRTXToKICAgICAgICAgICAgYnJhbmRfZWwgPSBjYXJkLnNlbGVjdF9vbmUoIi5wcm9kdWN0LWJyYW5kIikKICAgICAgICAgICAgbmFtZV9lbCA9IGNhcmQuc2VsZWN0X29uZSgiLnByb2R1Y3QtcHJvZHVjdCIpCiAgICAgICAgICAgIHByaWNlX2VsID0gY2FyZC5zZWxlY3Rfb25lKCIucHJvZHVjdC1kaXNjb3VudGVkUHJpY2UsIC5wcm9kdWN0LXByaWNlIikKICAgICAgICAgICAgbGlua19lbCA9IGNhcmQuc2VsZWN0X29uZSgiYSIpCgogICAgICAgICAgICBicmFuZCA9IGJyYW5kX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpIGlmIGJyYW5kX2VsIGVsc2UgIiIKICAgICAgICAgICAgbmFtZSA9IG5hbWVfZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkgaWYgbmFtZV9lbCBlbHNlICIiCiAgICAgICAgICAgIHRpdGxlID0gZiJ7YnJhbmR9IHtuYW1lfSIuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgdGl0bGU6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgcHJpY2UgPSAwCiAgICAgICAgICAgIGlmIHByaWNlX2VsOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHByaWNlID0gaW50KHJlLnN1YihyIlteXGRdIiwgIiIsIHByaWNlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpKSkKICAgICAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgICAgIHByb2RfdXJsID0gIiIKICAgICAgICAgICAgaWYgbGlua19lbCBhbmQgbGlua19lbC5nZXQoImhyZWYiKToKICAgICAgICAgICAgICAgIGhyZWYgPSBsaW5rX2VsWyJocmVmIl0KICAgICAgICAgICAgICAgIHByb2RfdXJsID0gaHJlZiBpZiBocmVmLnN0YXJ0c3dpdGgoImh0dHAiKSBlbHNlIGYiaHR0cHM6Ly93d3cubXludHJhLmNvbXtocmVmfSIKCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHsidGl0bGUiOiB0aXRsZSwgInByaWNlIjogcHJpY2UsICJ1cmwiOiBwcm9kX3VybH0pCgogICAgICAgIHJldHVybiByZXN1bHRzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBbXQoKCmRlZiBzZWFyY2hfaGFtYXJhbWFsbChxdWVyeTogc3RyKSAtPiBsaXN0OgogICAgIiIiU2VhcmNoIEhhbWFyYSBNYWxsLiBVUkwgcGF0dGVybiBhbmQgc2VsZWN0b3JzIG5lZWQgbGl2ZSB2ZXJpZmljYXRpb24uIiIiCiAgICB0cnk6CiAgICAgICAgdXJsID0gZiJodHRwczovL3d3dy5oYW1hcmFtYWxsLmNvbS9zZWFyY2g/cT17cXVvdGVfcGx1cyhxdWVyeSl9IgogICAgICAgIHJlc3AgPSByZXF1ZXN0cy5nZXQodXJsLCBoZWFkZXJzPUhFQURFUlNbImhhbWFyYW1hbGwiXSwgdGltZW91dD0xNSkKICAgICAgICByZXNwLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgICAgIHNvdXAgPSBCZWF1dGlmdWxTb3VwKHJlc3AudGV4dCwgImx4bWwiKQoKICAgICAgICByZXN1bHRzID0gW10KICAgICAgICAjIFRyeSBjb21tb24gZS1jb21tZXJjZSBzZWFyY2ggcmVzdWx0IHBhdHRlcm5zCiAgICAgICAgY2FyZHMgPSBzb3VwLnNlbGVjdCgiLnByb2R1Y3QtY2FyZCwgLnByb2R1Y3QtaXRlbSwgLnNlYXJjaC1yZXN1bHQtaXRlbSwgLnByb2R1Y3QtZ3JpZC1pdGVtIikKICAgICAgICBmb3IgY2FyZCBpbiBjYXJkc1s6VE9QX05fUkVTVUxUU106CiAgICAgICAgICAgIHRpdGxlX2VsID0gY2FyZC5zZWxlY3Rfb25lKCJoMiwgaDMsIC5wcm9kdWN0LXRpdGxlLCAucHJvZHVjdC1uYW1lLCBhLnByb2R1Y3QtbGluayIpCiAgICAgICAgICAgIHByaWNlX2VsID0gY2FyZC5zZWxlY3Rfb25lKCIucHJpY2UsIC5wcm9kdWN0LXByaWNlLCAuY3VycmVudC1wcmljZSIpCiAgICAgICAgICAgIGxpbmtfZWwgPSBjYXJkLnNlbGVjdF9vbmUoImEiKQoKICAgICAgICAgICAgaWYgbm90IHRpdGxlX2VsOgogICAgICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgICAgIHRpdGxlID0gdGl0bGVfZWwuZ2V0X3RleHQoc3RyaXA9VHJ1ZSkKICAgICAgICAgICAgcHJpY2UgPSAwCiAgICAgICAgICAgIGlmIHByaWNlX2VsOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHByaWNlID0gaW50KHJlLnN1YihyIlteXGRdIiwgIiIsIHByaWNlX2VsLmdldF90ZXh0KHN0cmlwPVRydWUpKSkKICAgICAgICAgICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgICAgIHByb2RfdXJsID0gIiIKICAgICAgICAgICAgaWYgbGlua19lbCBhbmQgbGlua19lbC5nZXQoImhyZWYiKToKICAgICAgICAgICAgICAgIGhyZWYgPSBsaW5rX2VsWyJocmVmIl0KICAgICAgICAgICAgICAgIHByb2RfdXJsID0gaHJlZiBpZiBocmVmLnN0YXJ0c3dpdGgoImh0dHAiKSBlbHNlIGYiaHR0cHM6Ly93d3cuaGFtYXJhbWFsbC5jb217aHJlZn0iCgogICAgICAgICAgICByZXN1bHRzLmFwcGVuZCh7InRpdGxlIjogdGl0bGUsICJwcmljZSI6IHByaWNlLCAidXJsIjogcHJvZF91cmx9KQoKICAgICAgICByZXR1cm4gcmVzdWx0cwogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gW10KCgpkZWYgc2VhcmNoX2FsbF9wbGF0Zm9ybXMocHJvZHVjdDogZGljdCwgc2tpcF9wbGF0Zm9ybTogc3RyKSAtPiBkaWN0OgogICAgIiIiU2VhcmNoIGFsbCBwbGF0Zm9ybXMgZXhjZXB0IHRoZSBzb3VyY2UuIFJldHVybnMgZGljdCBvZiBwbGF0Zm9ybSAtPiByZXN1bHRzIGxpc3QuIiIiCiAgICBzZWFyY2hfZnVuY3MgPSB7CiAgICAgICAgImFtYXpvbiI6IHNlYXJjaF9hbWF6b24sCiAgICAgICAgImZsaXBrYXJ0Ijogc2VhcmNoX2ZsaXBrYXJ0LAogICAgICAgICJteW50cmEiOiBzZWFyY2hfbXludHJhLAogICAgICAgICJoYW1hcmFtYWxsIjogc2VhcmNoX2hhbWFyYW1hbGwsCiAgICB9CgogICAgcXVlcnkgPSBwcm9kdWN0LmdldCgic2VhcmNoX3F1ZXJ5IiwgIiIpCiAgICByZXN1bHRzID0ge30KCiAgICBmb3IgcGxhdGZvcm0sIGZ1bmMgaW4gc2VhcmNoX2Z1bmNzLml0ZW1zKCk6CiAgICAgICAgaWYgcGxhdGZvcm0gPT0gc2tpcF9wbGF0Zm9ybToKICAgICAgICAgICAgY29udGludWUKICAgICAgICByZXN1bHRzW3BsYXRmb3JtXSA9IGZ1bmMocXVlcnkpCiAgICAgICAgdGltZS5zbGVlcChSRVFVRVNUX0RFTEFZKQoKICAgIHJldHVybiByZXN1bHRzCg==",
    "matcher.py": "IyBtYXRjaGVyLnB5IOKAlCBTY29yaW5nICYgbWF0Y2ggc2VsZWN0aW9uCgppbXBvcnQgdGltZQpmcm9tIGNvbmZpZyBpbXBvcnQgU0NPUkVfRVhBQ1QsIFNDT1JFX1NJTUlMQVIsIFJFUVVFU1RfREVMQVkKZnJvbSB1dGlscyBpbXBvcnQgbm9ybWFsaXplX3F1YW50aXR5LCBleHRyYWN0X3F1YW50aXR5X2Zyb21fdGl0bGUKCgpkZWYgc2NvcmVfcmVzdWx0KHNvdXJjZTogZGljdCwgcmVzdWx0OiBkaWN0KSAtPiBpbnQ6CiAgICAiIiJTY29yZSBhIHNpbmdsZSBzZWFyY2ggcmVzdWx0IGFnYWluc3QgdGhlIHNvdXJjZSBwcm9kdWN0IChtYXggMTAwKS4iIiIKICAgIHNjb3JlID0gMAoKICAgICMgUnVsZSAxIOKAlCBCcmFuZCBtYXRjaCAocmVxdWlyZWQgZ2F0ZSwgKzQwKQogICAgaWYgc291cmNlLmdldCgiYnJhbmQiKSBhbmQgc291cmNlWyJicmFuZCJdLmxvd2VyKCkgbm90IGluIHJlc3VsdC5nZXQoInRpdGxlIiwgIiIpLmxvd2VyKCk6CiAgICAgICAgcmV0dXJuIDAKICAgIHNjb3JlICs9IDQwCgogICAgIyBSdWxlIDIg4oCUIFF1YW50aXR5IG1hdGNoICgrMzUpCiAgICBzb3VyY2VfcXR5ID0gbm9ybWFsaXplX3F1YW50aXR5KHNvdXJjZS5nZXQoInF1YW50aXR5IiwgIiIpKQogICAgcmVzdWx0X3F0eSA9IGV4dHJhY3RfcXVhbnRpdHlfZnJvbV90aXRsZShyZXN1bHQuZ2V0KCJ0aXRsZSIsICIiKSkKICAgIGlmIHNvdXJjZV9xdHkgYW5kIHJlc3VsdF9xdHkgYW5kIHNvdXJjZV9xdHkgPT0gbm9ybWFsaXplX3F1YW50aXR5KHJlc3VsdF9xdHkpOgogICAgICAgIHNjb3JlICs9IDM1CgogICAgIyBSdWxlIDMg4oCUIEtleXdvcmQgb3ZlcmxhcCAoKzI1KQogICAgc291cmNlX2t3ID0gc2V0KHNvdXJjZS5nZXQoImtleXdvcmRzIiwgW10pKQogICAgcmVzdWx0X3dvcmRzID0gc2V0KHJlc3VsdC5nZXQoInRpdGxlIiwgIiIpLmxvd2VyKCkuc3BsaXQoKSkKICAgIGlmIHNvdXJjZV9rdzoKICAgICAgICBvdmVybGFwID0gbGVuKHNvdXJjZV9rdyAmIHJlc3VsdF93b3JkcykgLyBsZW4oc291cmNlX2t3KQogICAgICAgIHNjb3JlICs9IGludChvdmVybGFwICogMjUpCgogICAgcmV0dXJuIHNjb3JlCgoKZGVmIHBpY2tfYmVzdF9tYXRjaChzb3VyY2U6IGRpY3QsIHJlc3VsdHM6IGxpc3QpIC0+IGRpY3Q6CiAgICAiIiJTY29yZSBhbGwgcmVzdWx0cywgcmV0dXJuIGJlc3Qgd2l0aCBzY29yZSBhbmQgbWF0Y2hfdHlwZS4iIiIKICAgIGlmIG5vdCByZXN1bHRzOgogICAgICAgIHJldHVybiB7Im1hdGNoX3R5cGUiOiAibm90X2ZvdW5kIiwgInNjb3JlIjogMH0KCiAgICBiZXN0ID0gTm9uZQogICAgYmVzdF9zY29yZSA9IC0xCgogICAgZm9yIHIgaW4gcmVzdWx0czoKICAgICAgICBzID0gc2NvcmVfcmVzdWx0KHNvdXJjZSwgcikKICAgICAgICBpZiBzID4gYmVzdF9zY29yZToKICAgICAgICAgICAgYmVzdF9zY29yZSA9IHMKICAgICAgICAgICAgYmVzdCA9IHIKCiAgICBpZiBiZXN0IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsibWF0Y2hfdHlwZSI6ICJub3RfZm91bmQiLCAic2NvcmUiOiAwfQoKICAgIG1hdGNoID0gZGljdChiZXN0KQogICAgbWF0Y2hbInNjb3JlIl0gPSBiZXN0X3Njb3JlCiAgICBpZiBiZXN0X3Njb3JlID49IFNDT1JFX0VYQUNUOgogICAgICAgIG1hdGNoWyJtYXRjaF90eXBlIl0gPSAiZXhhY3QiCiAgICBlbGlmIGJlc3Rfc2NvcmUgPj0gU0NPUkVfU0lNSUxBUjoKICAgICAgICBtYXRjaFsibWF0Y2hfdHlwZSJdID0gInNpbWlsYXIiCiAgICBlbHNlOgogICAgICAgIG1hdGNoWyJtYXRjaF90eXBlIl0gPSAibm90X2ZvdW5kIgoKICAgIHJldHVybiBtYXRjaAoKCmRlZiBmYWxsYmFja19zZWFyY2goc291cmNlOiBkaWN0LCBwbGF0Zm9ybTogc3RyKSAtPiBkaWN0OgogICAgIiIiUmUtc2VhcmNoIHdpdGggcmVsYXhlZCBxdWVyaWVzIHdoZW4gYmVzdCBzY29yZSBpcyBub3RfZm91bmQuIiIiCiAgICBmcm9tIHNlYXJjaGVyIGltcG9ydCBzZWFyY2hfYW1hem9uLCBzZWFyY2hfZmxpcGthcnQsIHNlYXJjaF9teW50cmEsIHNlYXJjaF9oYW1hcmFtYWxsCgogICAgc2VhcmNoX2Z1bmNzID0gewogICAgICAgICJhbWF6b24iOiBzZWFyY2hfYW1hem9uLAogICAgICAgICJmbGlwa2FydCI6IHNlYXJjaF9mbGlwa2FydCwKICAgICAgICAibXludHJhIjogc2VhcmNoX215bnRyYSwKICAgICAgICAiaGFtYXJhbWFsbCI6IHNlYXJjaF9oYW1hcmFtYWxsLAogICAgfQogICAgZnVuYyA9IHNlYXJjaF9mdW5jcy5nZXQocGxhdGZvcm0pCiAgICBpZiBub3QgZnVuYzoKICAgICAgICByZXR1cm4geyJtYXRjaF90eXBlIjogInVuYXZhaWxhYmxlIn0KCiAgICBicmFuZCA9IHNvdXJjZS5nZXQoImJyYW5kIiwgIiIpCiAgICBrZXl3b3JkcyA9IHNvdXJjZS5nZXQoImtleXdvcmRzIiwgW10pCgogICAgIyBGYWxsYmFjayAxOiBicmFuZCArIHByb2R1Y3QgbmFtZSAobm8gcXVhbnRpdHkpCiAgICBxdWVyeTEgPSBmInticmFuZH0geycgJy5qb2luKGtleXdvcmRzWzo1XSl9Ii5zdHJpcCgpCiAgICBpZiBxdWVyeTE6CiAgICAgICAgdGltZS5zbGVlcChSRVFVRVNUX0RFTEFZKQogICAgICAgIHJlc3VsdHMgPSBmdW5jKHF1ZXJ5MSkKICAgICAgICBtYXRjaCA9IHBpY2tfYmVzdF9tYXRjaChzb3VyY2UsIHJlc3VsdHMpCiAgICAgICAgaWYgbWF0Y2hbIm1hdGNoX3R5cGUiXSAhPSAibm90X2ZvdW5kIjoKICAgICAgICAgICAgcmV0dXJuIG1hdGNoCgogICAgIyBGYWxsYmFjayAyOiBwcm9kdWN0IHR5cGUgKyBrZXkgaW5ncmVkaWVudCAobm8gYnJhbmQpCiAgICBxdWVyeTIgPSAiICIuam9pbihrZXl3b3Jkc1s6M10pLnN0cmlwKCkKICAgIGlmIHF1ZXJ5MjoKICAgICAgICB0aW1lLnNsZWVwKFJFUVVFU1RfREVMQVkpCiAgICAgICAgcmVzdWx0cyA9IGZ1bmMocXVlcnkyKQogICAgICAgIG1hdGNoID0gcGlja19iZXN0X21hdGNoKHNvdXJjZSwgcmVzdWx0cykKICAgICAgICBpZiBtYXRjaFsibWF0Y2hfdHlwZSJdICE9ICJub3RfZm91bmQiOgogICAgICAgICAgICByZXR1cm4gbWF0Y2gKCiAgICByZXR1cm4geyJtYXRjaF90eXBlIjogInVuYXZhaWxhYmxlIn0KCgpkZWYgbWF0Y2hfYWxsX3BsYXRmb3Jtcyhzb3VyY2U6IGRpY3QsIHNlYXJjaF9yZXN1bHRzOiBkaWN0KSAtPiBkaWN0OgogICAgIiIiRm9yIGVhY2ggcGxhdGZvcm0sIHBpY2sgYmVzdCBtYXRjaDsgZmFsbGJhY2sgaWYgbm90X2ZvdW5kLiIiIgogICAgbWF0Y2hlcyA9IHt9CiAgICBmb3IgcGxhdGZvcm0sIHJlc3VsdHMgaW4gc2VhcmNoX3Jlc3VsdHMuaXRlbXMoKToKICAgICAgICBtYXRjaCA9IHBpY2tfYmVzdF9tYXRjaChzb3VyY2UsIHJlc3VsdHMpCiAgICAgICAgaWYgbWF0Y2hbIm1hdGNoX3R5cGUiXSA9PSAibm90X2ZvdW5kIjoKICAgICAgICAgICAgbWF0Y2ggPSBmYWxsYmFja19zZWFyY2goc291cmNlLCBwbGF0Zm9ybSkKICAgICAgICBtYXRjaGVzW3BsYXRmb3JtXSA9IG1hdGNoCiAgICByZXR1cm4gbWF0Y2hlcwo=",
    "sheets.py": "IyBzaGVldHMucHkg4oCUIEdvb2dsZSBTaGVldHMgY29tbWlzc2lvbiBsb29rdXAKCmltcG9ydCBnc3ByZWFkCmZyb20gZ29vZ2xlLm9hdXRoMi5zZXJ2aWNlX2FjY291bnQgaW1wb3J0IENyZWRlbnRpYWxzCgpmcm9tIGNvbmZpZyBpbXBvcnQgU0hFRVRfTkFNRSwgU0hFRVRfVEFCLCBDT01NSVNTSU9OX0NPTF9DQVRFR09SWSwgQ09NTUlTU0lPTl9DT0xfUEVSQ0VOVAoKIyBNb2R1bGUtbGV2ZWwgY2FjaGUKX2NvbW1pc3Npb25fY2FjaGUgPSBOb25lCgoKZGVmIGxvYWRfY29tbWlzc2lvbl9zaGVldCgpIC0+IGRpY3Q6CiAgICAiIiJBdXRoZW50aWNhdGUgd2l0aCBnc3ByZWFkLCByZWFkIGNvbW1pc3Npb24gc2hlZXQsIHJldHVybiB7Y2F0ZWdvcnk6IHBlcmNlbnR9IGRpY3QuIiIiCiAgICBnbG9iYWwgX2NvbW1pc3Npb25fY2FjaGUKICAgIGlmIF9jb21taXNzaW9uX2NhY2hlIGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiBfY29tbWlzc2lvbl9jYWNoZQoKICAgIHNjb3BlcyA9IFsKICAgICAgICAiaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vYXV0aC9zcHJlYWRzaGVldHMucmVhZG9ubHkiLAogICAgICAgICJodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9hdXRoL2RyaXZlLnJlYWRvbmx5IiwKICAgIF0KICAgIGNyZWRzID0gQ3JlZGVudGlhbHMuZnJvbV9zZXJ2aWNlX2FjY291bnRfZmlsZSgKICAgICAgICAic2VydmljZV9hY2NvdW50Lmpzb24iLCBzY29wZXM9c2NvcGVzCiAgICApCiAgICBjbGllbnQgPSBnc3ByZWFkLmF1dGhvcml6ZShjcmVkcykKCiAgICBzaGVldCA9IGNsaWVudC5vcGVuKFNIRUVUX05BTUUpCiAgICB0YWIgPSBzaGVldC53b3Jrc2hlZXQoU0hFRVRfVEFCKQogICAgcm93cyA9IHRhYi5nZXRfYWxsX3ZhbHVlcygpCgogICAgY29tbWlzc2lvbl9tYXAgPSB7fQogICAgZm9yIHJvdyBpbiByb3dzWzE6XTogICMgU2tpcCBoZWFkZXIKICAgICAgICBpZiBsZW4ocm93KSA+IG1heChDT01NSVNTSU9OX0NPTF9DQVRFR09SWSwgQ09NTUlTU0lPTl9DT0xfUEVSQ0VOVCk6CiAgICAgICAgICAgIGNhdGVnb3J5ID0gcm93W0NPTU1JU1NJT05fQ09MX0NBVEVHT1JZXS5zdHJpcCgpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHBlcmNlbnQgPSBmbG9hdChyb3dbQ09NTUlTU0lPTl9DT0xfUEVSQ0VOVF0uc3RyaXAoKS5yZXBsYWNlKCIlIiwgIiIpKQogICAgICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgogICAgICAgICAgICAgICAgcGVyY2VudCA9IDAuMAogICAgICAgICAgICBpZiBjYXRlZ29yeToKICAgICAgICAgICAgICAgIGNvbW1pc3Npb25fbWFwW2NhdGVnb3J5XSA9IHBlcmNlbnQKCiAgICBfY29tbWlzc2lvbl9jYWNoZSA9IGNvbW1pc3Npb25fbWFwCiAgICByZXR1cm4gX2NvbW1pc3Npb25fY2FjaGUKCgpkZWYgZ2V0X2NvbW1pc3Npb24oY2F0ZWdvcnk6IHN0ciwgY29tbWlzc2lvbl9tYXA6IGRpY3QpIC0+IGZsb2F0OgogICAgIiIiTG9vayB1cCBjb21taXNzaW9uICUgZm9yIGEgY2F0ZWdvcnkuIFRyaWVzIGV4YWN0LCBjYXNlLWluc2Vuc2l0aXZlLCB0aGVuIHBhcnRpYWwgbWF0Y2guIiIiCiAgICBpZiBub3QgY2F0ZWdvcnkgb3Igbm90IGNvbW1pc3Npb25fbWFwOgogICAgICAgIHJldHVybiAwLjAKCiAgICAjIEV4YWN0IG1hdGNoCiAgICBpZiBjYXRlZ29yeSBpbiBjb21taXNzaW9uX21hcDoKICAgICAgICByZXR1cm4gY29tbWlzc2lvbl9tYXBbY2F0ZWdvcnldCgogICAgIyBDYXNlLWluc2Vuc2l0aXZlIG1hdGNoCiAgICBjYXRfbG93ZXIgPSBjYXRlZ29yeS5sb3dlcigpCiAgICBmb3Iga2V5LCB2YWwgaW4gY29tbWlzc2lvbl9tYXAuaXRlbXMoKToKICAgICAgICBpZiBrZXkubG93ZXIoKSA9PSBjYXRfbG93ZXI6CiAgICAgICAgICAgIHJldHVybiB2YWwKCiAgICAjIFBhcnRpYWwgbWF0Y2gKICAgIGZvciBrZXksIHZhbCBpbiBjb21taXNzaW9uX21hcC5pdGVtcygpOgogICAgICAgIGlmIGNhdF9sb3dlciBpbiBrZXkubG93ZXIoKSBvciBrZXkubG93ZXIoKSBpbiBjYXRfbG93ZXI6CiAgICAgICAgICAgIHJldHVybiB2YWwKCiAgICByZXR1cm4gMC4wCgoKZGVmIHJlZnJlc2hfY29tbWlzc2lvbl9jYWNoZSgpOgogICAgIiIiQ2xlYXIgY2FjaGUgYW5kIHJlbG9hZCBmcm9tIHNoZWV0LiIiIgogICAgZ2xvYmFsIF9jb21taXNzaW9uX2NhY2hlCiAgICBfY29tbWlzc2lvbl9jYWNoZSA9IE5vbmUKICAgIHJldHVybiBsb2FkX2NvbW1pc3Npb25fc2hlZXQoKQo=",
    "formatter.py": "IyBmb3JtYXR0ZXIucHkg4oCUIFRlbGVncmFtIG1lc3NhZ2UgYnVpbGRlcgoKZnJvbSBjb25maWcgaW1wb3J0IFBMQVRGT1JNX0VNT0pJLCBTQ09SRV9FWEFDVAoKCmRlZiBmb3JtYXRfcGxhdGZvcm1fcm93KHBsYXRmb3JtOiBzdHIsIG1hdGNoOiBkaWN0LCBzb3VyY2VfcGxhdGZvcm06IHN0cikgLT4gc3RyOgogICAgIiIiQnVpbGQgb25lIGxpbmUgb2YgdGhlIHByaWNlIGNvbXBhcmlzb24gdGFibGUuIiIiCiAgICBlbW9qaSA9IFBMQVRGT1JNX0VNT0pJLmdldChwbGF0Zm9ybSwgIuKdkyIpCiAgICBuYW1lID0gcGxhdGZvcm0uY2FwaXRhbGl6ZSgpCiAgICBpZiBwbGF0Zm9ybSA9PSAiaGFtYXJhbWFsbCI6CiAgICAgICAgbmFtZSA9ICJIYW1hcmEgTWFsbCIKCiAgICBzdWZmaXggPSAiICh5b3VyIGxpbmspIiBpZiBwbGF0Zm9ybSA9PSBzb3VyY2VfcGxhdGZvcm0gZWxzZSAiIgoKICAgIG1hdGNoX3R5cGUgPSBtYXRjaC5nZXQoIm1hdGNoX3R5cGUiLCAidW5hdmFpbGFibGUiKQogICAgaWYgbWF0Y2hfdHlwZSA9PSAidW5hdmFpbGFibGUiOgogICAgICAgIHJldHVybiBmIntlbW9qaX0ge25hbWU6PDEyfSDigJQgICAg4p2MIG5vdCBsaXN0ZWQiCiAgICBpZiBtYXRjaF90eXBlID09ICJub3RfZm91bmQiOgogICAgICAgIHJldHVybiBmIntlbW9qaX0ge25hbWU6PDEyfSDigJQgICAg4p2MIG5vdCBmb3VuZCIKCiAgICBwcmljZSA9IG1hdGNoLmdldCgicHJpY2UiLCAwKQogICAgcHJpY2Vfc3RyID0gZiLigrl7cHJpY2V9IiBpZiBwcmljZSBlbHNlICLigJQiCgogICAgaWYgbWF0Y2hfdHlwZSA9PSAiZXhhY3QiOgogICAgICAgIGluZGljYXRvciA9ICLinIUiCiAgICBlbGlmIG1hdGNoX3R5cGUgPT0gInNpbWlsYXIiOgogICAgICAgIGluZGljYXRvciA9ICLimqDvuI8gc2ltaWxhciIKICAgIGVsc2U6CiAgICAgICAgaW5kaWNhdG9yID0gIuKdjCIKCiAgICByZXR1cm4gZiJ7ZW1vaml9IHtuYW1lOjwxMn0ge3ByaWNlX3N0cjo8OH0ge2luZGljYXRvcn17c3VmZml4fSIKCgpkZWYgZmluZF9iZXN0X2RlYWwobWF0Y2hlczogZGljdCkgLT4gdHVwbGU6CiAgICAiIiJGaW5kIGNoZWFwZXN0IHBsYXRmb3JtIGFuZCBiZXN0IGVhcm5pbmcgcGxhdGZvcm0gYW1vbmcgZXhhY3QgbWF0Y2hlcy4KICAgIFJldHVybnMgKGNoZWFwZXN0X3BsYXRmb3JtLCBiZXN0X2Vhcm5pbmdfcGxhdGZvcm0pLiIiIgogICAgY2hlYXBlc3RfcGxhdGZvcm0gPSBOb25lCiAgICBjaGVhcGVzdF9wcmljZSA9IGZsb2F0KCJpbmYiKQogICAgYmVzdF9lYXJuaW5nX3BsYXRmb3JtID0gTm9uZQogICAgYmVzdF9lYXJuaW5nID0gMC4wCgogICAgZm9yIHBsYXRmb3JtLCBtYXRjaCBpbiBtYXRjaGVzLml0ZW1zKCk6CiAgICAgICAgaWYgbWF0Y2guZ2V0KCJtYXRjaF90eXBlIikgIT0gImV4YWN0IjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmljZSA9IG1hdGNoLmdldCgicHJpY2UiLCAwKQogICAgICAgIGlmIHByaWNlIGFuZCBwcmljZSA8IGNoZWFwZXN0X3ByaWNlOgogICAgICAgICAgICBjaGVhcGVzdF9wcmljZSA9IHByaWNlCiAgICAgICAgICAgIGNoZWFwZXN0X3BsYXRmb3JtID0gcGxhdGZvcm0KCiAgICByZXR1cm4gY2hlYXBlc3RfcGxhdGZvcm0sIGNoZWFwZXN0X3ByaWNlCgoKZGVmIGZvcm1hdF9mdWxsX21lc3NhZ2Uoc291cmNlOiBkaWN0LCBtYXRjaGVzOiBkaWN0LCBjb21taXNzaW9uOiBmbG9hdCkgLT4gc3RyOgogICAgIiIiQXNzZW1ibGUgdGhlIGNvbXBsZXRlIFRlbGVncmFtIG1lc3NhZ2UuIiIiCiAgICBicmFuZCA9IHNvdXJjZS5nZXQoImJyYW5kIiwgIiIpCiAgICBuYW1lID0gc291cmNlLmdldCgibmFtZSIsICIiKQogICAgcXVhbnRpdHkgPSBzb3VyY2UuZ2V0KCJxdWFudGl0eSIsICIiKQogICAgY2F0ZWdvcnkgPSBzb3VyY2UuZ2V0KCJjYXRlZ29yeSIsICIiKQogICAgc291cmNlX3BsYXRmb3JtID0gc291cmNlLmdldCgicGxhdGZvcm0iLCAiIikKCiAgICAjIEhlYWRlcgogICAgbGluZXMgPSBbCiAgICAgICAgZiLwn5SNIHticmFuZH0ge25hbWV9IHtxdWFudGl0eX0iLnN0cmlwKCksCiAgICAgICAgZiLwn5OCIHtjYXRlZ29yeX0gfCDwn5K4IEVhcm5LYXJvIENvbW1pc3Npb246IHtjb21taXNzaW9ufSUiIGlmIGNvbW1pc3Npb24gZWxzZSBmIvCfk4Ige2NhdGVnb3J5fSIsCiAgICAgICAgIiIsCiAgICAgICAgIvCfkrAgUHJpY2UgQ29tcGFyaXNvbiIsCiAgICAgICAgIuKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgeKUgSIsCiAgICBdCgogICAgIyBTb3VyY2UgcGxhdGZvcm0gcm93IGZpcnN0CiAgICBpZiBzb3VyY2VfcGxhdGZvcm06CiAgICAgICAgc291cmNlX21hdGNoID0gewogICAgICAgICAgICAibWF0Y2hfdHlwZSI6ICJleGFjdCIsCiAgICAgICAgICAgICJwcmljZSI6IHNvdXJjZS5nZXQoInByaWNlIiwgMCksCiAgICAgICAgfQogICAgICAgIGxpbmVzLmFwcGVuZChmb3JtYXRfcGxhdGZvcm1fcm93KHNvdXJjZV9wbGF0Zm9ybSwgc291cmNlX21hdGNoLCBzb3VyY2VfcGxhdGZvcm0pKQoKICAgICMgT3RoZXIgcGxhdGZvcm1zCiAgICBwbGF0Zm9ybV9vcmRlciA9IFsiYW1hem9uIiwgImZsaXBrYXJ0IiwgIm15bnRyYSIsICJoYW1hcmFtYWxsIl0KICAgIGZvciBwIGluIHBsYXRmb3JtX29yZGVyOgogICAgICAgIGlmIHAgPT0gc291cmNlX3BsYXRmb3JtOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIHAgaW4gbWF0Y2hlczoKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGZvcm1hdF9wbGF0Zm9ybV9yb3cocCwgbWF0Y2hlc1twXSwgc291cmNlX3BsYXRmb3JtKSkKCiAgICAjIEJlc3QgZGVhbAogICAgY2hlYXBlc3RfcGxhdGZvcm0sIGNoZWFwZXN0X3ByaWNlID0gZmluZF9iZXN0X2RlYWwobWF0Y2hlcykKICAgICMgQWxzbyBjaGVjayBzb3VyY2UgcHJpY2UKICAgIHNvdXJjZV9wcmljZSA9IHNvdXJjZS5nZXQoInByaWNlIiwgMCkKICAgIGlmIHNvdXJjZV9wcmljZSBhbmQgKG5vdCBjaGVhcGVzdF9wbGF0Zm9ybSBvciBzb3VyY2VfcHJpY2UgPCBjaGVhcGVzdF9wcmljZSk6CiAgICAgICAgY2hlYXBlc3RfcGxhdGZvcm0gPSBzb3VyY2VfcGxhdGZvcm0KICAgICAgICBjaGVhcGVzdF9wcmljZSA9IHNvdXJjZV9wcmljZQoKICAgIGlmIGNoZWFwZXN0X3BsYXRmb3JtOgogICAgICAgIGxpbmVzLmFwcGVuZCgiIikKICAgICAgICBwX25hbWUgPSAiSGFtYXJhIE1hbGwiIGlmIGNoZWFwZXN0X3BsYXRmb3JtID09ICJoYW1hcmFtYWxsIiBlbHNlIGNoZWFwZXN0X3BsYXRmb3JtLmNhcGl0YWxpemUoKQogICAgICAgIGxpbmVzLmFwcGVuZChmIvCfj4YgTG93ZXN0IHByaWNlOiB7cF9uYW1lfSDigrl7Y2hlYXBlc3RfcHJpY2V9IikKCiAgICAjIEJlc3QgZWFybmluZwogICAgaWYgY29tbWlzc2lvbiBhbmQgY2hlYXBlc3RfcGxhdGZvcm06CiAgICAgICAgZWFybmluZyA9IHJvdW5kKGNoZWFwZXN0X3ByaWNlICogY29tbWlzc2lvbiAvIDEwMCwgMikKICAgICAgICBsaW5lcy5hcHBlbmQoZiLwn5KhIEJlc3QgZWFybmluZzoge3BfbmFtZX0g4oaSIOKCuXtlYXJuaW5nfSIpCgogICAgIyBTaW1pbGFyIHdhcm5pbmcKICAgIGhhc19zaW1pbGFyID0gYW55KG0uZ2V0KCJtYXRjaF90eXBlIikgPT0gInNpbWlsYXIiIGZvciBtIGluIG1hdGNoZXMudmFsdWVzKCkpCiAgICBpZiBoYXNfc2ltaWxhcjoKICAgICAgICBsaW5lcy5hcHBlbmQoIiIpCiAgICAgICAgbGluZXMuYXBwZW5kKCLimqDvuI8gTm90ZTogU2ltaWxhciBwcm9kdWN0cyBtYXkgZGlmZmVyIGluIHNpemUvdmFyaWFudCIpCgogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykK",
    "bot.py": "IyBib3QucHkg4oCUIE1haW4gVGVsZWdyYW0gYm90IGVudHJ5IHBvaW50CgppbXBvcnQgb3MKaW1wb3J0IHRyYWNlYmFjawpmcm9tIHRlbGVncmFtIGltcG9ydCBVcGRhdGUKZnJvbSB0ZWxlZ3JhbS5leHQgaW1wb3J0IEFwcGxpY2F0aW9uLCBDb21tYW5kSGFuZGxlciwgTWVzc2FnZUhhbmRsZXIsIGZpbHRlcnMsIENvbnRleHRUeXBlcwoKZnJvbSB1dGlscyBpbXBvcnQgZGV0ZWN0X3BsYXRmb3JtCmZyb20gZXh0cmFjdG9yIGltcG9ydCBleHRyYWN0X3Byb2R1Y3QKZnJvbSBzZWFyY2hlciBpbXBvcnQgc2VhcmNoX2FsbF9wbGF0Zm9ybXMKZnJvbSBtYXRjaGVyIGltcG9ydCBtYXRjaF9hbGxfcGxhdGZvcm1zCmZyb20gc2hlZXRzIGltcG9ydCBsb2FkX2NvbW1pc3Npb25fc2hlZXQsIGdldF9jb21taXNzaW9uCmZyb20gZm9ybWF0dGVyIGltcG9ydCBmb3JtYXRfZnVsbF9tZXNzYWdlCgojIE1vZHVsZS1sZXZlbCBjb21taXNzaW9uIG1hcCDigJQgbG9hZGVkIGF0IHN0YXJ0dXAKY29tbWlzc2lvbl9tYXAgPSB7fQoKU1RBUlRfTUVTU0FHRSA9ICIiIvCfkYsgV2VsY29tZSB0byBFYXJuS2FybyBQcmljZSBCb3QhCgpTZW5kIG1lIGFueSBwcm9kdWN0IGxpbmsgZnJvbToKLSBBbWF6b24uaW4KLSBGbGlwa2FydAotIE15bnRyYQotIEhhbWFyYSBNYWxsCgpJJ2xsIGNvbXBhcmUgcHJpY2VzIGFjcm9zcyBhbGwgcGxhdGZvcm1zIGFuZCBzaG93IHlvdXIgRWFybkthcm8gY29tbWlzc2lvbiBpbnN0YW50bHkuIiIiCgoKYXN5bmMgZGVmIHN0YXJ0X2NvbW1hbmQodXBkYXRlOiBVcGRhdGUsIGNvbnRleHQ6IENvbnRleHRUeXBlcy5ERUZBVUxUX1RZUEUpOgogICAgIiIiSGFuZGxlIC9zdGFydCBjb21tYW5kLiIiIgogICAgYXdhaXQgdXBkYXRlLm1lc3NhZ2UucmVwbHlfdGV4dChTVEFSVF9NRVNTQUdFKQoKCmFzeW5jIGRlZiBoYW5kbGVfbWVzc2FnZSh1cGRhdGU6IFVwZGF0ZSwgY29udGV4dDogQ29udGV4dFR5cGVzLkRFRkFVTFRfVFlQRSk6CiAgICAiIiJNYWluIGhhbmRsZXIg4oCUIHJ1bnMgb24gZXZlcnkgdXNlciBtZXNzYWdlLiIiIgogICAgdXJsID0gdXBkYXRlLm1lc3NhZ2UudGV4dC5zdHJpcCgpCgogICAgIyBTdGVwIDEg4oCUIFZhbGlkYXRlIGl0J3MgYSBrbm93biBwbGF0Zm9ybSBVUkwKICAgIHRyeToKICAgICAgICBwbGF0Zm9ybSA9IGRldGVjdF9wbGF0Zm9ybSh1cmwpCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICBhd2FpdCB1cGRhdGUubWVzc2FnZS5yZXBseV90ZXh0KAogICAgICAgICAgICAi4pqg77iPIFBsZWFzZSBzZW5kIGEgcHJvZHVjdCBsaW5rIGZyb20gQW1hem9uLCBGbGlwa2FydCwgTXludHJhLCBvciBIYW1hcmEgTWFsbC4iCiAgICAgICAgKQogICAgICAgIHJldHVybgoKICAgICMgU3RlcCAyIOKAlCBBY2tub3dsZWRnZSBpbW1lZGlhdGVseQogICAgYXdhaXQgdXBkYXRlLm1lc3NhZ2UucmVwbHlfdGV4dCgi4o+zIFNlYXJjaGluZyBhY3Jvc3MgcGxhdGZvcm1zLCBnaXZlIG1lIGEgbW9tZW50Li4uIikKCiAgICB0cnk6CiAgICAgICAgIyBTdGVwIDMg4oCUIEV4dHJhY3QgcHJvZHVjdCBmcm9tIHNvdXJjZSBVUkwKICAgICAgICBwcm9kdWN0ID0gZXh0cmFjdF9wcm9kdWN0KHVybCkKCiAgICAgICAgaWYgbm90IHByb2R1Y3QuZ2V0KCJuYW1lIik6CiAgICAgICAgICAgIGF3YWl0IHVwZGF0ZS5tZXNzYWdlLnJlcGx5X3RleHQoCiAgICAgICAgICAgICAgICAi4p2MIENvdWxkbid0IGV4dHJhY3QgcHJvZHVjdCBkZXRhaWxzIGZyb20gdGhhdCBsaW5rLiBQbGVhc2UgdHJ5IGEgZGlmZmVyZW50IG9uZS4iCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmV0dXJuCgogICAgICAgICMgU3RlcCA0IOKAlCBTZWFyY2ggYWxsIG90aGVyIHBsYXRmb3JtcwogICAgICAgIHNlYXJjaF9yZXN1bHRzID0gc2VhcmNoX2FsbF9wbGF0Zm9ybXMocHJvZHVjdCwgc2tpcF9wbGF0Zm9ybT1wbGF0Zm9ybSkKCiAgICAgICAgIyBTdGVwIDUg4oCUIE1hdGNoIGFuZCBzY29yZQogICAgICAgIG1hdGNoZXMgPSBtYXRjaF9hbGxfcGxhdGZvcm1zKHByb2R1Y3QsIHNlYXJjaF9yZXN1bHRzKQoKICAgICAgICAjIFN0ZXAgNiDigJQgR2V0IGNvbW1pc3Npb24KICAgICAgICBjb21taXNzaW9uID0gZ2V0X2NvbW1pc3Npb24ocHJvZHVjdC5nZXQoImNhdGVnb3J5IiwgIiIpLCBjb21taXNzaW9uX21hcCkKCiAgICAgICAgIyBTdGVwIDcg4oCUIEZvcm1hdCBhbmQgc2VuZAogICAgICAgIG1lc3NhZ2UgPSBmb3JtYXRfZnVsbF9tZXNzYWdlKHByb2R1Y3QsIG1hdGNoZXMsIGNvbW1pc3Npb24pCiAgICAgICAgYXdhaXQgdXBkYXRlLm1lc3NhZ2UucmVwbHlfdGV4dChtZXNzYWdlKQoKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICB0cmFjZWJhY2sucHJpbnRfZXhjKCkKICAgICAgICBhd2FpdCB1cGRhdGUubWVzc2FnZS5yZXBseV90ZXh0KAogICAgICAgICAgICAi4p2MIFNvbWV0aGluZyB3ZW50IHdyb25nLiBQbGVhc2UgdHJ5IGFnYWluIG9yIHNlbmQgYSBkaWZmZXJlbnQgbGluay4iCiAgICAgICAgKQoKCmFzeW5jIGRlZiBlcnJvcl9oYW5kbGVyKHVwZGF0ZTogb2JqZWN0LCBjb250ZXh0OiBDb250ZXh0VHlwZXMuREVGQVVMVF9UWVBFKToKICAgICIiIkNhdGNoIGFsbCB1bmhhbmRsZWQgZXhjZXB0aW9ucy4iIiIKICAgIHByaW50KGYiRXJyb3I6IHtjb250ZXh0LmVycm9yfSIpCiAgICB0cmFjZWJhY2sucHJpbnRfZXhjKCkKCgpkZWYgbWFpbigpOgogICAgIiIiTG9hZCBjb21taXNzaW9uIG1hcCwgYnVpbGQgYm90LCBzdGFydCBwb2xsaW5nLiIiIgogICAgZ2xvYmFsIGNvbW1pc3Npb25fbWFwCgogICAgIyBMb2FkIC5lbnYgZmlsZSBpZiBwcmVzZW50CiAgICBlbnZfcGF0aCA9IG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUoX19maWxlX18pIG9yICIuIiwgIi5lbnYiKQogICAgaWYgb3MucGF0aC5leGlzdHMoZW52X3BhdGgpOgogICAgICAgIHdpdGggb3BlbihlbnZfcGF0aCkgYXMgZjoKICAgICAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgICAgIGlmIGxpbmUgYW5kIG5vdCBsaW5lLnN0YXJ0c3dpdGgoIiMiKSBhbmQgIj0iIGluIGxpbmU6CiAgICAgICAgICAgICAgICAgICAga2V5LCB2YWwgPSBsaW5lLnNwbGl0KCI9IiwgMSkKICAgICAgICAgICAgICAgICAgICBvcy5lbnZpcm9uLnNldGRlZmF1bHQoa2V5LnN0cmlwKCksIHZhbC5zdHJpcCgpKQoKICAgIHRva2VuID0gb3MuZW52aXJvbi5nZXQoIlRFTEVHUkFNX0JPVF9UT0tFTiIsICIiKQogICAgaWYgbm90IHRva2VuOgogICAgICAgIHByaW50KCLinYwgVEVMRUdSQU1fQk9UX1RPS0VOIG5vdCBzZXQuIFVzZSB0aGUgdG9rZW4gc2V0dXAgY2VsbCBvciBjcmVhdGUgYSAuZW52IGZpbGUuIikKICAgICAgICByZXR1cm4KCiAgICAjIExvYWQgY29tbWlzc2lvbnMKICAgIHRyeToKICAgICAgICBjb21taXNzaW9uX21hcCA9IGxvYWRfY29tbWlzc2lvbl9zaGVldCgpCiAgICAgICAgcHJpbnQoZiLinIUgTG9hZGVkIHtsZW4oY29tbWlzc2lvbl9tYXApfSBjb21taXNzaW9uIGNhdGVnb3JpZXMiKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYi4pqg77iPIENvdWxkIG5vdCBsb2FkIGNvbW1pc3Npb24gc2hlZXQ6IHtlfSIpCiAgICAgICAgcHJpbnQoIiAgIEJvdCB3aWxsIHJ1biB3aXRob3V0IGNvbW1pc3Npb24gZGF0YS4iKQoKICAgIGFwcCA9IEFwcGxpY2F0aW9uLmJ1aWxkZXIoKS50b2tlbih0b2tlbikuYnVpbGQoKQoKICAgICMgUmVnaXN0ZXIgaGFuZGxlcnMKICAgIGFwcC5hZGRfaGFuZGxlcihDb21tYW5kSGFuZGxlcigic3RhcnQiLCBzdGFydF9jb21tYW5kKSkKICAgIGFwcC5hZGRfaGFuZGxlcihNZXNzYWdlSGFuZGxlcihmaWx0ZXJzLlRFWFQgJiB+ZmlsdGVycy5DT01NQU5ELCBoYW5kbGVfbWVzc2FnZSkpCiAgICBhcHAuYWRkX2Vycm9yX2hhbmRsZXIoZXJyb3JfaGFuZGxlcikKCiAgICBwcmludCgi8J+kliBCb3QgaXMgcnVubmluZy4uLiBTZW5kIGl0IGEgcHJvZHVjdCBsaW5rIG9uIFRlbGVncmFtIikKICAgIGFwcC5ydW5fcG9sbGluZygpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
}

for fname, b64_content in files_b64.items():
    with open(fname, "wb") as f:
        f.write(base64.b64decode(b64_content))
    print(f"  ✅ {fname}")

print(f"\n✅ All {len(files_b64)} files written to {os.getcwd()}")


In [ ]:
# === Cell 4: Upload service_account.json (SKIP if not using Google Sheets) ===
from google.colab import files as colab_files
import os

if not os.path.exists("service_account.json"):
    print("📤 Upload your service_account.json file:")
    uploaded = colab_files.upload()
else:
    print("✅ service_account.json already exists")

In [ ]:
# === Cell 5: Enter Telegram Bot Token ===
import os
from getpass import getpass

token = getpass("Enter your TELEGRAM_BOT_TOKEN: ")
os.environ["TELEGRAM_BOT_TOKEN"] = token
print("✅ Token saved")

In [ ]:
# === Cell 6: Run Unit Tests (Optional) ===
import importlib
import config, utils, matcher
for m in [config, utils, matcher]:
    importlib.reload(m)

from utils import detect_platform, normalize_quantity, extract_quantity_from_title, clean_price, extract_keywords
from matcher import score_result, pick_best_match

# utils tests
assert detect_platform("https://www.amazon.in/dp/B08XYZ") == "amazon"
assert detect_platform("https://www.flipkart.com/some-product") == "flipkart"
assert normalize_quantity("250 ML") == "250ml"
assert normalize_quantity("1 KG") == "1000g"
assert clean_price("₹1,299") == 1299
assert clean_price("Rs. 1299.00") == 1299
assert extract_quantity_from_title("Mamaearth Onion Hair Oil 250ml") == "250ml"
print("✅ utils.py OK")

# matcher tests
source = {"brand": "Mamaearth", "quantity": "250ml", "keywords": ["onion", "hair", "oil", "growth", "control"]}
assert score_result(source, {"title": "Mamaearth Onion Hair Oil 250ml", "price": 299}) >= 75
assert score_result(source, {"title": "WOW Onion Hair Oil 250ml", "price": 280}) == 0
print("✅ matcher.py OK")

print("\n🎉 All tests passed!")

In [ ]:
# === Cell 7: Keep-Alive ===
from IPython.display import display, Javascript

display(Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
'''))
print("⏰ Keep-alive active")

In [ ]:
# === Cell 8: START THE BOT ===
# ⚠️ This cell BLOCKS while the bot runs. Stop with ⏹ button.

import importlib
import config, utils, extractor, searcher, matcher, sheets, formatter, bot
for mod in [config, utils, extractor, searcher, matcher, sheets, formatter, bot]:
    importlib.reload(mod)

bot.main()

---
## 🔄 Session Restart: Run cells 1 → 2 → 5 → 8